# AttnFool evaluation

Evaluates the seven classifiers used in the paper on a 1000-image subset of
ImageNet-2012 val, plus DETR on a 100-image subset of MS COCO 2017 val.

Classifiers: **ResNet50, ViT-T, ViT-B, ViT-B/384, DeiT-T, DeiT-B, DeiT-B/384**.

Before running, build the data subsets:

```bash
python -m datasets.prepare_data --imagenet-src /path/to/ImageNet/val
```

Knobs to tune cost: `NUM_IMAGENET`, `NUM_COCO`, `ATTACK_ITERS`, `BATCH_SIZE`,
`MODELS_TO_RUN`.

In [1]:
%pip install numpy
%pip install matplotlib
%pip install scipy
%pip install scikit-learn
%pip install torch
%pip install torchvision
%pip install torchaudio
%pip install transformers
%pip install datasets
%pip install wandb
%pip install timm
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, sys, math, time, json, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from PIL import Image

REPO = Path('.').resolve()
sys.path.insert(0, str(REPO))

from models.DeiT import (
    deit_tiny_patch16_224, deit_base_patch16_224, deit_base_patch16_384,
)
from models.vision_transformer import (
    vit_tiny_patch16_224, vit_base_patch16_224, vit_base_patch16_384,
)
from models.resnet import ResNet50
from utils import (
    apply_patch, attn_fool_loss, build_patch_mask, cosine_step_size,
    normalized_momentum, patch_token_index, mu as IMNET_MU, std as IMNET_STD,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

ModuleNotFoundError: No module named 'models.DeiT'

## Config

In [ ]:
NUM_IMAGENET = 1000          # images sampled by datasets.prepare_data
NUM_COCO     = 100

BATCH_SIZE   = 16            # drop if you OOM on the 384 models
ATTACK_ITERS = 250           # paper default; lower to e.g. 50 for a quick run
ATTACK_LR    = 8.0 / 255.0
ATTACK_MODE  = 'AttnFool_kq' # 'CE_loss' | 'AttnFool_kq' | 'AttnFool_kqstar'
ATTN_W       = 1.0
USE_MOMENTUM = False
PATCH_SIZE   = 16
PATCH_ROW    = 0
PATCH_COL    = 0
SEED         = 0

MODELS_TO_RUN = [
    'ResNet50',
    'ViT-T', 'ViT-B', 'ViT-B-384',
    'DeiT-T', 'DeiT-B', 'DeiT-B-384',
]

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

## Model registry

Each entry returns `(model, is_transformer, img_size)`. The transformer
flag controls whether we pull `(q,k)` from the model for the AttnFool
loss.

In [ ]:
def _resnet50():
    m = ResNet50(pretrained=True)
    return m, False, 224

MODEL_FACTORY = {
    'ResNet50'  : _resnet50,
    'ViT-T'     : lambda: (vit_tiny_patch16_224(pretrained=True), True, 224),
    'ViT-B'     : lambda: (vit_base_patch16_224(pretrained=True), True, 224),
    'ViT-B-384' : lambda: (vit_base_patch16_384(pretrained=True), True, 384),
    'DeiT-T'    : lambda: (deit_tiny_patch16_224(pretrained=True), True, 224),
    'DeiT-B'    : lambda: (deit_base_patch16_224(pretrained=True), True, 224),
    'DeiT-B-384': lambda: (deit_base_patch16_384(pretrained=True), True, 384),
}

def forward(model, x, is_transformer):
    if is_transformer:
        out, qk_list = model(x)
        return out, qk_list
    return model(x), None

## ImageNet val loader (1000 images)

In [ ]:
IMAGENET_DIR = REPO / 'data' / 'imagenet_val'
assert IMAGENET_DIR.is_dir(), (
    f'Missing {IMAGENET_DIR}. Run `python -m datasets.prepare_data --imagenet-src <path>` first.'
)

def make_imagenet_loader(img_size, batch_size=BATCH_SIZE):
    resize = int(img_size / 0.875) if img_size == 224 else img_size
    tfm = transforms.Compose([
        transforms.Resize(resize, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMNET_MU, std=IMNET_STD),
    ])
    ds = torchvision.datasets.ImageFolder(str(IMAGENET_DIR), transform=tfm)
    print(f'  imagenet({img_size}): {len(ds)} images, {len(ds.classes)} classes')
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False,
                                       num_workers=2, pin_memory=True)

# AttentionFool Implementation

A from-scratch, self-contained implementation of the Attention-Fool patch
attack (Lovisotto et al.). The cells below build the attack bottom-up:

1. **`ce_loss`** — the standard cross-entropy objective `L_ce`, and why it is
   not enough on its own.
2. **`pgd_patch_attack`** — projected gradient descent that crafts the patch
   in `[0,1]` pixel space with a cosine-decayed signed step.
3. **`single_head_kq_loss`** — `L^{hl}_{kq}`, the single-head / single-layer
   building block: pull every query toward the patch key.
4. **`l_kq`** — `L_kq`, the same loss aggregated across all heads and layers
   with a smooth maximum (log-sum-exp).
5. **`l_kq_star`** — `L_kq*`, the class-token-focused variant for ViT/DeiT.

Each loss returns a scalar that PGD **maximizes**. The attention losses read
the model's **pre-softmax** `QK^T/√d` logits (`qk_list`), not the post-softmax
attention weights.


In [ ]:
# =============================================================================
#  L_ce -- cross-entropy loss (the standard model-output objective)
# =============================================================================
#
#       L_ce(z, y) = -log( softmax(z)_y )
#
#   z = model logits, y = correct class label.
#
# How PGD uses it:
#   - UNTARGETED attack: MAXIMIZE L_ce(z, y_true). Pushing CE up makes the
#     model less confident in the correct class.
#   - TARGETED attack:   MAXIMIZE -L_ce(z, y_target), which is the same as
#     MINIMIZING CE toward the chosen target class.
#
# Why L_ce alone is limited (the paper's motivation for Attention-Fool):
#   PGD driven by L_ce alone tends to exploit gradients flowing through the
#   VALUE path of attention, while largely ignoring the more dangerous
#   vulnerability in the attention WEIGHTS themselves -- gradients through the
#   softmax attention weights are comparatively tiny. The L_kq / L_kq* losses
#   (below) attack the attention logits directly to expose that vulnerability.
# -----------------------------------------------------------------------------


def ce_loss(logits, y, targeted=False, y_target=None):
    """Cross-entropy attack term.

    Args:
        logits: model output logits, shape [batch, n_classes].
        y: true labels [batch] (used for the untargeted attack).
        targeted: if True, return -CE toward y_target instead.
        y_target: target labels [batch], required when targeted=True.

    Returns:
        Scalar loss that PGD MAXIMIZES.
    """
    if targeted:
        assert y_target is not None, 'targeted attack needs y_target'
        # Maximizing -CE(z, y_target) == minimizing CE toward the target.
        return -F.cross_entropy(logits, y_target)
    # Untargeted: maximizing CE(z, y_true) erodes confidence in the true class.
    return F.cross_entropy(logits, y)


In [ ]:
# =============================================================================
#  PGD -- projected gradient descent for the adversarial patch
# =============================================================================
#
# PGD is the optimizer that crafts the patch. The update (paper, Eq. for p):
#
#       p_{t+1} = clamp_{[0,1]}( p_t + alpha_t * sign( grad_p L ) )
#
#   - p is the patch, optimized in raw PIXEL space [0,1] (NOT normalized space)
#   - alpha_t is a cosine-decayed step size starting at alpha_0 = 8/255
#   - L is the attack loss (CE, CE+L_kq, or CE+L_kq*)
#   - we MAXIMIZE L, hence "+ sign(grad)" (gradient *ascent*)
#
# Pixel-space workflow each step:
#   1. paste patch into the [0,1] image, then apply ImageNet normalization
#   2. forward the normalized image, capturing the pre-softmax QK logits
#   3. compute the chosen attack loss
#   4. backprop to the patch pixels, take a signed step, clamp to [0,1]
#
# NOTE: this cell calls `l_kq` / `l_kq_star`, which are DEFINED IN THE CELLS
# BELOW. That is fine -- Python resolves them when pgd_patch_attack() actually
# RUNS, by which point every cell has been executed. Run the cells top-to-
# bottom once before calling this function.
# -----------------------------------------------------------------------------

# ImageNet normalization stats (kept local so this section is self-contained).
PGD_MEAN = (0.485, 0.456, 0.406)
PGD_STD  = (0.229, 0.224, 0.225)


def build_patch_mask(shape, patch_size, patch_row, patch_col, device):
    """Binary mask (1 inside the patch, 0 elsewhere), shape [B,C,H,W]."""
    mask = torch.zeros(shape, device=device)
    mask[:, :, patch_row:patch_row + patch_size,
               patch_col:patch_col + patch_size] = 1.0
    return mask


def patch_token_index(patch_row, patch_col, patch_size=16, img_size=224,
                      has_cls_token=True):
    """Map a pixel-space patch location to its key-token index i*.

    For a CLS-token model with a (img_size/patch_size) grid of patch tokens,
    the top-left patch (row=0, col=0) lands at token index 1 (CLS is 0).
    """
    grid = img_size // patch_size
    idx = (patch_row // patch_size) * grid + (patch_col // patch_size)
    return idx + (1 if has_cls_token else 0)


def cosine_step_size(alpha_0, t, n_iters):
    """Cosine-decayed step size: alpha_0 at t=0 -> ~0 at t=n_iters."""
    return alpha_0 * 0.5 * (1.0 + math.cos(math.pi * t / n_iters))


def _paste_and_normalize(x_norm, patch_01, mask, mu, std):
    """Paste the [0,1] patch into the image, then re-apply normalization.

    x_norm is the already-normalized clean image; we de-normalize, splice in
    the patch over the masked region, then normalize again so the model sees a
    valid input.
    """
    x_01 = x_norm * std + mu                       # back to [0,1]
    x_01 = patch_01 * mask + x_01 * (1 - mask)     # splice patch into image
    return (x_01 - mu) / std                        # re-normalize


def pgd_patch_attack(model, x, y, *,
                     is_transformer=True,
                     loss_type='ce_plus_kq',      # 'ce' | 'ce_plus_kq' | 'ce_plus_kq_star'
                     attn_w=1.0,
                     patch_size=16, patch_row=0, patch_col=0, img_size=224,
                     target_key_index=None, target_query_index=0,
                     n_iters=250, alpha_0=8.0 / 255.0,
                     use_momentum=False, beta=0.9,
                     targeted=False, y_target=None):
    """Craft an adversarial patch for a single batch with PGD.

    Args:
        model: the classifier. For transformers it must return (logits, qk_list)
               where qk_list is a per-layer list of pre-softmax [B,H,Q,K] tensors.
        x: normalized input batch [B,3,H,W].
        y: true labels [B].
        loss_type: which attack objective to maximize.
        target_key_index: i*; if None it is computed from the patch location.
        target_query_index: j* for L_kq* (CLS token, default 0).
    Returns:
        (x_adv, patch_01): the final adversarial (normalized) image and patch.
    """
    # Freeze the model -- we only optimize the patch pixels.
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    device = x.device
    mu  = torch.tensor(PGD_MEAN, device=device).view(1, 3, 1, 1)
    std = torch.tensor(PGD_STD,  device=device).view(1, 3, 1, 1)
    mask = build_patch_mask(x.shape, patch_size, patch_row, patch_col, device)

    if target_key_index is None and is_transformer:
        target_key_index = patch_token_index(patch_row, patch_col,
                                              patch_size, img_size)

    # Initialize the patch ~ U(0,1); outside the mask we keep the real image.
    x_01 = x * std + mu
    patch_01 = torch.rand_like(x) * mask + x_01 * (1 - mask)
    patch_01 = patch_01.detach().requires_grad_(True)
    momentum = torch.zeros_like(patch_01) if use_momentum else None

    for t in range(n_iters):
        if patch_01.grad is not None:
            patch_01.grad = None

        # 1-2. paste + normalize, then forward (capture QK logits).
        x_adv = _paste_and_normalize(x, patch_01, mask, mu, std)
        if is_transformer:
            logits, qk_list = model(x_adv)
        else:
            logits, qk_list = model(x_adv), None

        # 3. assemble the attack loss.
        loss = ce_loss(logits, y, targeted=targeted, y_target=y_target)
        if is_transformer and loss_type == 'ce_plus_kq':
            loss = loss + attn_w * l_kq(qk_list, target_key_index)
        elif is_transformer and loss_type == 'ce_plus_kq_star':
            loss = loss + attn_w * l_kq_star(qk_list, target_query_index,
                                             target_key_index)
        # (loss_type == 'ce' uses CE alone)

        # 4. backprop to the patch pixels.
        grad = torch.autograd.grad(loss, patch_01)[0]

        # Optional normalized momentum, else raw gradient sign.
        if use_momentum:
            g = grad / (grad.flatten(1).norm(dim=1).view(-1, 1, 1, 1) + 1e-12)
            momentum = beta * momentum + (1 - beta) * g
            direction = momentum.sign()
        else:
            direction = grad.sign()

        # Signed ascent step (we MAXIMIZE), then project back to [0,1].
        alpha_t = cosine_step_size(alpha_0, t, n_iters)
        with torch.no_grad():
            patch_01 = patch_01 + alpha_t * direction * mask
            patch_01 = patch_01.clamp(0, 1)
            # Keep the unmasked region pinned to the real image.
            patch_01 = patch_01 * mask + x_01 * (1 - mask)
        patch_01.requires_grad_(True)

    with torch.no_grad():
        x_adv = _paste_and_normalize(x, patch_01, mask, mu, std)
    return x_adv, patch_01.detach()


In [ ]:
# =============================================================================
#  L^{hl}_{kq}  -- single-head, single-layer Attention-Fool loss
# =============================================================================
#
# This is the basic building block of Attention-Fool.
#
#       L^{hl}_{kq} = (1/n) * sum_j  B^{hl}_{j, i*}
#
# where B^{hl}_{j,i} is the PRE-softmax dot-product similarity between query
# token j and key token i (i.e. (Q K^T)/sqrt(d_k)) for head h in layer l.
#
# Exact recipe (one layer l, one head h):
#   1. Choose the layer l and head h            -> caller passes in B_hl.
#   2. Identify the patch key token i*          -> target_key_index.
#   3. Take COLUMN i* of B_hl: B_hl[:, :, i*]   -> every query's similarity
#                                                  to the patch key.
#   4. Average that column over all queries (and the batch).
#   5. PGD MAXIMIZES this loss.
#
# Intuition: we are telling the model "make every query token look at the
# patch token." We never ask it to predict a wrong class directly -- instead
# we corrupt the internal attention routing so that many queries attend to the
# malicious key (the paper's Figure 2).
#
# IMPORTANT: use the PRE-softmax logits B, not the post-softmax attention
# weights. The gradient through softmax weights is tiny; the routing
# vulnerability lives in the raw similarities.
# -----------------------------------------------------------------------------


def single_head_kq_loss(B_hl, target_key_index):
    """L^{hl}_{kq} for ONE head in ONE layer.

    Args:
        B_hl: pre-softmax similarity matrix for one head,
              shape [batch, n_query, n_key]. B_hl[b, j, i] is the similarity
              between query j and key i.
        target_key_index: i*  -- the patch key token index.

    Returns:
        Scalar = mean over queries (and batch) of column i*.
    """
    # Column i*: similarity of EVERY query j to the patch key i*.
    col = B_hl[..., target_key_index]   # [batch, n_query]
    # (1/n) * sum_j  -> average over queries; .mean() also averages the batch.
    return col.mean()


In [ ]:
# =============================================================================
#  L_kq  -- all-heads, all-layers Attention-Fool loss
# =============================================================================
#
# A single head is rarely enough to break the model, so we aggregate the
# single-head losses L^{hl}_{kq} across every head and every layer.
#
# The paper does NOT use a plain average. A plain average would reward
# spreading the perturbation weakly across many heads. Instead it uses a
# SMOOTH MAXIMUM (log-sum-exp):
#
#       L^l_kq = log  sum_h exp( L^{hl}_kq )        (over heads in layer l)
#       L_kq   = log  sum_l exp( L^l_kq   )         (over layers)
#
# log-sum-exp behaves like max(...) but stays smooth/differentiable, so the
# gradient naturally concentrates on the most vulnerable heads/layers --
# exactly where a few successful attention failures matter most.
# -----------------------------------------------------------------------------


def l_kq(qk_list, target_key_index):
    """Full L_kq: log-sum-exp over heads, then log-sum-exp over layers.

    Mirrors COMPUTE_L_KQ in the spec.

    Args:
        qk_list: list (len = #layers) of pre-softmax B tensors, each
                 [batch, n_heads, n_query, n_key].
        target_key_index: i*  -- the patch key token index.

    Returns:
        Scalar loss to MAXIMIZE.
    """
    layer_scores = []
    for B in qk_list:                               # for every layer l
        n_heads = B.shape[1]
        head_scores = []
        for h in range(n_heads):                    # for every head h
            B_hl = B[:, h]                          # [batch, n_query, n_key]
            head_scores.append(single_head_kq_loss(B_hl, target_key_index))
        head_scores = torch.stack(head_scores)      # [n_heads]
        # Smooth maximum over heads -> one score for this layer.
        layer_scores.append(torch.logsumexp(head_scores, dim=0))
    layer_scores = torch.stack(layer_scores)        # [n_layers]
    # Smooth maximum over layers -> single scalar.
    return torch.logsumexp(layer_scores, dim=0)


In [ ]:
# =============================================================================
#  L_kq*  -- class-token-focused Attention-Fool loss
# =============================================================================
#
# L_kq (previous cell) pulls EVERY query toward the patch key. L_kq* is more
# surgical: it targets ONLY the query that actually drives the prediction.
#
# In ViT / DeiT the final class logits are read off the CLS token, so the CLS
# token is the single most important query. If we make the CLS query attend to
# the adversarial patch key, we corrupt the global decision directly.
#
#   Single head / layer:   L^{hl}_{kq*} = B_{j*, i*}
#       j* = target_query_index  (CLS token index, usually 0 for ViT/DeiT)
#       i* = target_key_index    (patch key token index, usually 1 for the
#                                 top-left patch when CLS sits at index 0)
#
#   Across heads / layers we reuse the SAME smooth-max (log-sum-exp)
#   aggregation as L_kq, so the attack concentrates on the most vulnerable
#   head/layer rather than spreading itself thin.
# -----------------------------------------------------------------------------


def single_head_kq_star_loss(B_hl, target_query_index, target_key_index):
    """L^{hl}_{kq*} for ONE head in ONE layer.

    Args:
        B_hl: pre-softmax similarity matrix for one head,
              shape [batch, n_query, n_key].
        target_query_index: j*  -- the class-token query index.
        target_key_index:   i*  -- the patch key token index.

    Returns:
        Scalar = B_{j*, i*}, averaged over the batch.
    """
    # A single scalar per image: similarity of the CLS query to the patch key.
    score = B_hl[:, target_query_index, target_key_index]   # -> [batch]
    return score.mean()                                     # mean over batch


def l_kq_star(qk_list, target_query_index, target_key_index):
    """Full L_kq*: log-sum-exp over heads, then over layers.

    Mirrors COMPUTE_L_KQ_STAR in the spec.

    Args:
        qk_list: list (len = #layers) of pre-softmax B tensors, each
                 [batch, n_heads, n_query, n_key].
        target_query_index: j*  (e.g. 0 for the CLS token).
        target_key_index:   i*  (patch key token index).

    Returns:
        Scalar loss to MAXIMIZE.
    """
    layer_scores = []
    for B in qk_list:                                   # for every layer l
        n_heads = B.shape[1]
        head_scores = torch.stack([
            single_head_kq_star_loss(B[:, h], target_query_index, target_key_index)
            for h in range(n_heads)                     # for every head h
        ])
        # Smooth maximum over heads -> one score per layer.
        layer_scores.append(torch.logsumexp(head_scores, dim=0))
    # Smooth maximum over layers -> single scalar.
    return torch.logsumexp(torch.stack(layer_scores), dim=0)


## AttnFool attack (per batch)

Mirrors `main.py`. The patch is initialized uniformly in `[0,1]` at
`(PATCH_ROW, PATCH_COL)` and refined with PGD on cosine-decayed step size.

In [ ]:
def attack_batch(model, is_transformer, X, y, network_name,
                 attack_mode=ATTACK_MODE, n_iters=ATTACK_ITERS,
                 lr=ATTACK_LR, atten_w=ATTN_W, use_momentum=USE_MOMENTUM,
                 patch_size=PATCH_SIZE, patch_row=PATCH_ROW, patch_col=PATCH_COL,
                 img_size=224):
    mu_t  = torch.tensor(IMNET_MU).view(1, 3, 1, 1).to(X.device)
    std_t = torch.tensor(IMNET_STD).view(1, 3, 1, 1).to(X.device)
    criterion = nn.CrossEntropyLoss()
    mask = build_patch_mask(X.shape, patch_size, patch_row, patch_col, X.device)

    target_token_idx = (
        patch_token_index(network_name, patch_row, patch_col,
                          patch_size=patch_size, img_size=img_size)
        if is_transformer else 0
    )

    x_01 = X * std_t + mu_t
    patch_01 = torch.rand_like(X) * mask + x_01 * (1 - mask)
    patch_01 = patch_01.detach().requires_grad_(True)
    m_state = torch.zeros_like(patch_01) if use_momentum else None

    for t in range(n_iters):
        if patch_01.grad is not None:
            patch_01.grad = None
        x_adv = apply_patch(X, patch_01, mask, mu_t, std_t)
        out, qk_list = forward(model, x_adv, is_transformer)
        loss = criterion(out, y)
        if is_transformer and attack_mode == 'AttnFool_kq':
            loss = loss + atten_w * attn_fool_loss(qk_list, target_token_idx, None)
        elif is_transformer and attack_mode == 'AttnFool_kqstar':
            loss = loss + atten_w * attn_fool_loss(qk_list, target_token_idx, 0)
        grad = torch.autograd.grad(loss, patch_01)[0]

        if use_momentum:
            m_state = normalized_momentum(m_state, grad, beta=0.9)
            direction = m_state.sign()
        else:
            direction = grad.sign()

        alpha_t = cosine_step_size(lr, t, n_iters)
        with torch.no_grad():
            patch_01 = patch_01 + alpha_t * direction * mask
            patch_01 = patch_01.clamp(0, 1)
            patch_01 = patch_01 * mask + x_01 * (1 - mask)
        patch_01.requires_grad_(True)

    with torch.no_grad():
        x_adv = apply_patch(X, patch_01, mask, mu_t, std_t)
        out, _ = forward(model, x_adv, is_transformer)
    return out

## Run the classifier evaluation

In [ ]:
import subprocess, os
from pathlib import Path

RESULTS_PATH = Path('results.json')
results = json.loads(RESULTS_PATH.read_text()) if RESULTS_PATH.exists() else {}

def run_models(names):
    """Run each model in its own subprocess so CUDA memory is fully released
    on process exit. Merges results into `results` and persists results.json
    after each model so a crash mid-group doesn't lose earlier wins."""
    for name in names:
        print(f'\n=== {name} ===', flush=True)
        cmd = [
            sys.executable, '-u', 'run_model.py',
            '--name', name,
            '--num-imagenet', str(NUM_IMAGENET),
            '--batch-size', str(BATCH_SIZE),
            '--attack-iters', str(ATTACK_ITERS),
            '--attack-lr', str(ATTACK_LR),
            '--attack-mode', ATTACK_MODE,
            '--attn-w', str(ATTN_W),
            '--patch-size', str(PATCH_SIZE),
            '--patch-row', str(PATCH_ROW),
            '--patch-col', str(PATCH_COL),
            '--seed', str(SEED),
        ]
        if USE_MOMENTUM:
            cmd.append('--use-momentum')

        env = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, env=env, cwd=str(REPO), bufsize=1)
        last_json = None
        for line in proc.stdout:
            line = line.rstrip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict) and 'clean_acc' in obj:
                    last_json = obj
                    continue
            except json.JSONDecodeError:
                pass
            print(line, flush=True)
        proc.wait()
        if proc.returncode != 0 or last_json is None:
            print(f'  [{name}] FAILED (exit {proc.returncode})', flush=True)
            continue
        results[name] = {'clean_acc': last_json['clean_acc'],
                         'attnfool_acc': last_json['attnfool_acc']}
        RESULTS_PATH.write_text(json.dumps(results, indent=2))
    return results


### Group 1: ResNet50, ViT-T, ViT-B (224)

In [ ]:
run_models(['ResNet50', 'ViT-T', 'ViT-B'])
print(json.dumps(results, indent=2))

### Group 2: DeiT-T, DeiT-B (224)

In [ ]:
run_models(['DeiT-T', 'DeiT-B'])
print(json.dumps(results, indent=2))

### Group 3: ViT-B-384, DeiT-B-384 (384, heaviest)

In [ ]:
run_models(['ViT-B-384', 'DeiT-B-384'])
print('\nSummary'); print(json.dumps(results, indent=2))

## DETR on COCO val subset

In [ ]:
from models.detr import build_detr, detect, COCO_CLASSES

COCO_DIR = REPO / 'datasets' / 'coco_val'
assert COCO_DIR.is_dir(), 'Run datasets/prepare_data.py to populate datasets/coco_val first.'

detr = build_detr('detr_resnet50', pretrained=True).to(DEVICE).eval()

img_paths = sorted((COCO_DIR / 'images').glob('*.jpg'))[:NUM_COCO]
print(f'COCO subset: {len(img_paths)} images')

with open(COCO_DIR / 'instances_val2017_subset.json') as f:
    gt = json.load(f)
gt_by_file = {im['file_name']: im for im in gt['images']}
gt_per_img = {}
for a in gt['annotations']:
    gt_per_img.setdefault(a['image_id'], []).append(a)

per_image = []
for p in img_paths:
    img = Image.open(p).convert('RGB')
    dets = detect(detr, img, score_threshold=0.7, device=DEVICE)
    gt_meta = gt_by_file.get(p.name, {})
    per_image.append({
        'file'    : p.name,
        'n_dets'  : len(dets),
        'n_gt'    : len(gt_per_img.get(gt_meta.get('id'), [])),
        'labels'  : sorted({d['label'] for d in dets}),
    })

avg_dets = sum(r['n_dets'] for r in per_image) / max(1, len(per_image))
avg_gt   = sum(r['n_gt']   for r in per_image) / max(1, len(per_image))
print(f'\nAvg DETR detections / image (score>0.7): {avg_dets:.2f}')
print(f'Avg ground-truth boxes / image          : {avg_gt:.2f}')
for r in per_image[:5]:
    print(r)

### Visualize one DETR prediction

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

img = Image.open(img_paths[0]).convert('RGB')
dets = detect(detr, img, score_threshold=0.7, device=DEVICE)
fig, ax = plt.subplots(figsize=(8, 8)); ax.imshow(img); ax.axis('off')
for d in dets:
    x0, y0, x1, y1 = d['box_xyxy']
    ax.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, linewidth=2, edgecolor='lime'))
    ax.text(x0, y0, f"{d['label']} {d['score']:.2f}", color='black',
            bbox=dict(facecolor='lime', alpha=0.6, pad=1))
plt.show()